# Machine Learning：從資料整理到回歸與分類


主要欄位：

| 欄位 | 說明 |
|---|---|
| session_id | Session 編號 |
| age | 年齡 |
| monthly_income | 月收入 |
| device | 裝置 |
| traffic_source | 流量來源 |
| campaign | 行銷活動 |
| segment | 客群 |
| session_hour | Session 開始時間 |
| is_weekend | 是否週末 |
| page_view | 瀏覽次數 |
| add_to_cart | 加入購物車次數 |
| purchase_amount | 購買金額 |
| purchased | 是否購買，0=沒有，1=有 |

In [71]:
import pandas as pd
pd.set_option("display.max_columns", 100)
df = pd.read_csv("baseline_ml_data.csv")

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 3012 entries, 0 to 3011
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   session_id       3012 non-null   str    
 1   age              2937 non-null   float64
 2   monthly_income   2928 non-null   float64
 3   device           2983 non-null   str    
 4   traffic_source   2979 non-null   str    
 5   campaign         2773 non-null   str    
 6   segment          2950 non-null   str    
 7   session_hour     3012 non-null   int64  
 8   is_weekend       3012 non-null   int64  
 9   page_view        3012 non-null   int64  
 10  add_to_cart      3012 non-null   int64  
 11  purchase_amount  3012 non-null   float64
 12  purchased        3012 non-null   int64  
dtypes: float64(3), int64(5), str(5)
memory usage: 387.2 KB


但是在真正建立模型之前，通常要先確認：

```text
資料有沒有缺失？
有沒有重複？
資料型別正不正確？
類別欄位有哪些值？
```

In [72]:
print("缺失量")
print(df.isna().sum().sort_values(ascending=False))

缺失量
campaign           239
monthly_income      84
age                 75
segment             62
traffic_source      33
device              29
session_id           0
session_hour         0
is_weekend           0
page_view            0
add_to_cart          0
purchase_amount      0
purchased            0
dtype: int64


In [73]:
print("重複數量:",df.duplicated().sum())

重複數量: 12


In [74]:
for column in  ["device", "traffic_source", "campaign", "segment"]:
    print(f"\n{column}:")
    print(df[column].value_counts())


device:
device
mobile     1674
desktop    1051
tablet      258
Name: count, dtype: int64

traffic_source:
traffic_source
organic     998
ads         935
referral    541
email       505
Name: count, dtype: int64

campaign:
campaign
none           1439
search          346
vip_offer       341
brand           337
summer_sale     310
Name: count, dtype: int64

segment:
segment
new       1486
growth    1017
vip        447
Name: count, dtype: int64


In [75]:
df = df.drop_duplicates()
print("重複數量:",df.duplicated().sum())
print("清理後資料筆數:",len(df))


重複數量: 0
清理後資料筆數: 3000


In [76]:
# import pandas as pd
# df = pd.DataFrame({
#     'name': ['Tom', 'Amy', 'Tom', 'Jack', 'Tom'],
#     'age':  [20, 18, 20, 25, 20]
# })
# print(df)
# print(df.duplicated())

## 處理缺失值


In [77]:
# 數值型欄位
numeric_missing_columns = [
    "age",
    "monthly_income"
]
for column in numeric_missing_columns:
    df[column] = df[column].fillna(df[column].median())
print(df.info())    

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   session_id       3000 non-null   str    
 1   age              3000 non-null   float64
 2   monthly_income   3000 non-null   float64
 3   device           2971 non-null   str    
 4   traffic_source   2967 non-null   str    
 5   campaign         2761 non-null   str    
 6   segment          2938 non-null   str    
 7   session_hour     3000 non-null   int64  
 8   is_weekend       3000 non-null   int64  
 9   page_view        3000 non-null   int64  
 10  add_to_cart      3000 non-null   int64  
 11  purchase_amount  3000 non-null   float64
 12  purchased        3000 non-null   int64  
dtypes: float64(3), int64(5), str(5)
memory usage: 386.0 KB
None


In [78]:
# 類別型欄位
categorical_missing_columns = [
    "device",
    "traffic_source",
    "campaign",
    "segment"
]
for column in categorical_missing_columns:
    df[column] = df[column].fillna("未知")
print(df.info())    

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   session_id       3000 non-null   str    
 1   age              3000 non-null   float64
 2   monthly_income   3000 non-null   float64
 3   device           3000 non-null   str    
 4   traffic_source   3000 non-null   str    
 5   campaign         3000 non-null   str    
 6   segment          3000 non-null   str    
 7   session_hour     3000 non-null   int64  
 8   is_weekend       3000 non-null   int64  
 9   page_view        3000 non-null   int64  
 10  add_to_cart      3000 non-null   int64  
 11  purchase_amount  3000 non-null   float64
 12  purchased        3000 non-null   int64  
dtypes: float64(3), int64(5), str(5)
memory usage: 386.6 KB
None


## Feature 與 Target
Feature 是：
> **給模型看的資料**

## Target

Target 是：

> **希望模型預測的答案**

## One-Hot Encoding

mobile
desktop
ads
organic
vip
new

# One-Hot Encoding

像下面這些文字：

```text
mobile
desktop
ads
organic
vip
new
```

In [79]:
model_data = df.drop(columns=["session_id"])
model_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              3000 non-null   float64
 1   monthly_income   3000 non-null   float64
 2   device           3000 non-null   str    
 3   traffic_source   3000 non-null   str    
 4   campaign         3000 non-null   str    
 5   segment          3000 non-null   str    
 6   session_hour     3000 non-null   int64  
 7   is_weekend       3000 non-null   int64  
 8   page_view        3000 non-null   int64  
 9   add_to_cart      3000 non-null   int64  
 10  purchase_amount  3000 non-null   float64
 11  purchased        3000 non-null   int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 345.2 KB


In [80]:
model_data = pd.get_dummies(model_data,
                             columns=[
        "device",
        "traffic_source",
        "campaign",
        "segment"
    ],dtype=int)
print(model_data.head())

    age  monthly_income  session_hour  is_weekend  page_view  add_to_cart  \
0  34.0         48887.0            10           0          5            1   
1  36.0         77288.0            19           0          7            0   
2  22.0         38869.0            10           1          4            0   
3  28.0         38820.0            15           0          5            0   
4  30.0         76136.0            16           0          3            0   

   purchase_amount  purchased  device_desktop  device_mobile  device_tablet  \
0              0.0          0               0              0              0   
1              0.0          0               0              1              0   
2              0.0          0               0              1              0   
3              0.0          0               0              1              0   
4              0.0          0               0              1              0   

   device_未知  traffic_source_ads  traffic_source_email  \
0   

In [81]:
# 查看 One-Hot Encoding 產生的部分欄位
one_hot_columns = [
    column
    for column in model_data.columns
    if column.startswith(
        (
            "device_",
            "traffic_source_",
            "campaign_",
            "segment_"
        )
    )
]

print(one_hot_columns)

['device_desktop', 'device_mobile', 'device_tablet', 'device_未知', 'traffic_source_ads', 'traffic_source_email', 'traffic_source_organic', 'traffic_source_referral', 'traffic_source_未知', 'campaign_brand', 'campaign_none', 'campaign_search', 'campaign_summer_sale', 'campaign_vip_offer', 'campaign_未知', 'segment_growth', 'segment_new', 'segment_vip', 'segment_未知']


In [82]:
regression_data = model_data.loc[model_data["purchased"] == 1]
print("purchased:",len(regression_data))

purchased: 413


In [ ]:
reg_feature_columns =  regression_data.columns.tolist()
reg_feature_columns.remove("purchased")
reg_feature_columns.remove("purchase_amount")
X = regression_data[reg_feature_columns]
y = regression_data["purchase_amount"]

['age', 'monthly_income', 'session_hour', 'is_weekend', 'page_view', 'add_to_cart', 'purchase_amount', 'purchased', 'device_desktop', 'device_mobile', 'device_tablet', 'device_未知', 'traffic_source_ads', 'traffic_source_email', 'traffic_source_organic', 'traffic_source_referral', 'traffic_source_未知', 'campaign_brand', 'campaign_none', 'campaign_search', 'campaign_summer_sale', 'campaign_vip_offer', 'campaign_未知', 'segment_growth', 'segment_new', 'segment_vip', 'segment_未知']


In [84]:
print("X:",X.shape)
print("y:",y.shape)

X: (413, 25)
y: (413,)


In [85]:
from sklearn.model_selection import train_test_split
# test_size=0.2 20%拿來測試
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((330, 25), (83, 25), (330,), (83,))

## StandardScaler：標準化

In [86]:
from sklearn.preprocessing import StandardScaler

# 需要標準化的數值欄位
numeric_feature_columns = [
    "age",
    "monthly_income",
    "session_hour",
    "is_weekend",
    "page_view",
    "add_to_cart"
]
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scaler = StandardScaler()
#再進行轉換標準化
X_train_scaled[numeric_feature_columns] = scaler.fit_transform(X_train[numeric_feature_columns])
X_test_scaled[numeric_feature_columns] = scaler.fit_transform(X_test[numeric_feature_columns])
X_train_scaled.shape,X_test_scaled.shape,y_train.shape,y_test.shape

((330, 25), (83, 25), (330,), (83,))

# 開始回歸模型

In [87]:
from sklearn.linear_model import LinearRegression
regression_model = LinearRegression()
regression_model.fit(X_train_scaled,y_train)
re_pred =  regression_model.predict(X_test_scaled)
print("預測結果:",re_pred[:5])

預測結果: [ 882.70003616  851.55569938 1819.13105148 1283.18450067 1001.91830886]


In [89]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test,re_pred)
print("MSE:",mse)


MSE: 53147.31340847798


## 分類模型

## Logistic Regression


> **這個 Session 會不會購買？**

答案只有：

```text
0 = 沒有購買
1 = 有購買
```

In [90]:
# 先取得所有欄位名稱
classification_feature_columns = model_data.columns.tolist()

# 移除不可以當作 Feature 的欄位
classification_feature_columns.remove("purchase_amount")
classification_feature_columns.remove("purchased")

# X 是模型輸入的 Feature
X = model_data[classification_feature_columns]

# y 是模型要預測的答案
y = model_data["purchased"]

# 查看資料大小
print("X：", X.shape)

# 查看 0 / 1 各有多少筆
print("\n0 / 1 數量：")
print(y.value_counts())

X： (3000, 25)

0 / 1 數量：
purchased
0    2587
1     413
Name: count, dtype: int64


# 將分類資料切成訓練集與測試集

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,    
    stratify=y#讓 Training 與 Testing 中的 0 / 1 比例盡量接近。
)

print("訓練資料：", X_train.shape)
print("測試資料：", X_test.shape)

print("\n訓練集購買比例：")
print(round(y_train.mean(), 4))

print("\n測試集購買比例：")
print(round(y_test.mean(), 4))

訓練資料： (2400, 25)
測試資料： (600, 25)

訓練集購買比例：
0.1375

測試集購買比例：
0.1383


# 標準化

In [ ]:
# 複製分類資料
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# 建立新的分類用 StandardScaler
classification_scaler = StandardScaler()

# 訓練資料：學習規則並轉換
X_train_scaled[numeric_feature_columns] = classification_scaler.fit_transform(
    X_train[numeric_feature_columns]
)

# 測試資料：只套用訓練資料的規則
X_test_scaled[numeric_feature_columns] = classification_scaler.transform(
    X_test[numeric_feature_columns]
)

# 建立 Logistic Regression

In [95]:
from sklearn.linear_model import LogisticRegression

# 建立 Logistic Regression 模型
classification_model = LogisticRegression(
    max_iter=1000
)

# 使用訓練資料建立模型
classification_model.fit(
    X_train_scaled,
    y_train
)

# 預測測試資料
classification_predictions = classification_model.predict(
    X_test_scaled
)

print("前 20 筆預測：")
print(classification_predictions[:20])

前 20 筆預測：
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
